# Notebook 05 — HistGradientBoosting

## Objectif
Tester un modèle non linéaire capable de mieux exploiter les interactions entre variables et de gérer nativement les valeurs manquantes. Comparer avec Ridge sur les mêmes folds GroupKFold.

## Fichiers utilisés
- `data/X_train.csv` — variables explicatives
- `data/y_train.csv` — variable cible

## Résultat attendu
- RMSE moyenne GroupKFold pour Ridge vs HGBR
- Rapport Skore Hub (clé `04_hgbr`)

## Notebook suivant
`06_tabular_pipeline.ipynb` — intégration des variables catégorielles avec skrub.

## 1. Imports

In [15]:
import json
import os
from pathlib import Path

import pandas as pd
import skore
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline

## 2. Connexion à Skore Hub

In [16]:
config = json.loads(Path("../.skore").read_text())
os.environ["SKORE_HUB_API_KEY"] = config["api_key"]
skore.login()
hub_project = skore.Project(
    "bobathon-esilv",
    mode="hub",
    workspace=config["workspace"],
)
print(f"Connecté au Hub : workspace={hub_project.workspace}")

╭───────────────────────────────── Login to Skore Hub ─────────────────────────────────╮
│                                                                                      │
│                                  Already logged in.                                  │
│                                                                                      │
╰──────────────────────────────────────────────────────────────────────────────────────╯

Connecté au Hub : workspace=BERNIE MADOFF


## 3. Chargement des données

In [17]:
DATA_DIR = Path("../data")

X_train = pd.read_csv(DATA_DIR / "X_train.csv")
y_train = pd.read_csv(DATA_DIR / "y_train.csv")

visits = X_train.merge(
    y_train,
    on="Index",
    how="inner",
    validate="one_to_one",
)

assert len(visits) == len(X_train)
assert visits["target"].notna().all()

print(f"visits : {visits.shape} — OK")

visits : (44590, 14) — OK


## 4. Préparation de X, y et des folds GroupKFold

In [18]:
feature_cols = [
    "sexM",
    "age_at_diagnosis",
    "age",
    "ledd",
    "time_since_intake_on",
    "time_since_intake_off",
    "on",
    "off",
]

X = visits[feature_cols]
y = visits["target"]
groups = visits["patient_id"]

cv_splits = list(GroupKFold(n_splits=5).split(X, y, groups=groups))

print(f"X : {X.shape}, {groups.nunique()} patients, {len(cv_splits)} folds")

X : (44590, 8), 5576 patients, 5 folds


## 5. Définition des modèles

`HistGradientBoostingRegressor` gère nativement les valeurs manquantes — pas besoin d'imputation.

In [19]:
ridge = make_pipeline(
    SimpleImputer(strategy="median"),
    Ridge(alpha=0.1),
)

hgbr = HistGradientBoostingRegressor(random_state=0)

print(f"ridge : {ridge}")
print(f"hgbr  : {hgbr}")

ridge : Pipeline(steps=[('simpleimputer', SimpleImputer(strategy='median')),
                ('ridge', Ridge(alpha=0.1))])
hgbr  : HistGradientBoostingRegressor(random_state=0)


## 6. Évaluation Ridge (référence)

In [20]:
report_ridge = skore.evaluate(ridge, X, y, splitter=cv_splits)
rmse_ridge = float(report_ridge.metrics.rmse().values.flatten().mean())
print(f"RMSE Ridge (GroupKFold, moyenne) : {rmse_ridge:.4f}")

Output()

RMSE Ridge (GroupKFold, moyenne) : 5.4317


## 7. Évaluation HGBR

In [21]:
report_hgbr = skore.evaluate(hgbr, X, y, splitter=cv_splits)
rmse_hgbr = float(report_hgbr.metrics.rmse().values.flatten().mean())
print(f"RMSE HGBR  (GroupKFold, moyenne) : {rmse_hgbr:.4f}")

Output()

RMSE HGBR  (GroupKFold, moyenne) : 3.9533


## 8. Tableau comparatif

In [22]:
rmse_ridge_df = report_ridge.metrics.rmse()
rmse_hgbr_df  = report_hgbr.metrics.rmse()

# Afficher la structure pour comprendre le format retourné
print('Shape Ridge:', rmse_ridge_df.shape)
print('Index Ridge:', rmse_ridge_df.index.tolist())
print('Columns Ridge:', rmse_ridge_df.columns.tolist())
print(rmse_ridge_df)
print()

# Extraire les valeurs par fold (lignes) pour chaque modèle
ridge_vals = rmse_ridge_df.iloc[:, 0].values   # 1 seule colonne : RMSE
hgbr_vals  = rmse_hgbr_df.iloc[:, 0].values

n_folds = len(ridge_vals)
comparison = pd.DataFrame({
    "Ridge": ridge_vals,
    "HGBR":  hgbr_vals,
}, index=[f"Fold {i+1}" for i in range(n_folds)])
comparison.loc["Moyenne"]    = comparison.mean()
comparison.loc["Écart-type"] = comparison.iloc[:n_folds].std()

print("RMSE par fold (GroupKFold) :")
display(comparison.round(4))

Shape Ridge: (1, 2)
Index Ridge: ['RMSE']
Columns Ridge: [('Ridge', 'mean'), ('Ridge', 'std')]
Estimator      Ridge          
Aggregate       mean       std
Metric                        
RMSE       10.596518  0.266972

RMSE par fold (GroupKFold) :


,Ridge,HGBR
Fold 1,10.5965,7.752
Moyenne,10.5965,7.752
Écart-type,NaN,NaN


## 9. Envoi des rapports Skore Hub

In [23]:
hub_project.put("04_hgbr", report_hgbr)
print("Rapport envoyé à Skore Hub sous la clé : 04_hgbr")

Output()

Consult your report at https://skore.probabl.ai/BERNIE 
MADOFF/bobathon-esilv/cross-validations/61221

Rapport envoyé à Skore Hub sous la clé : 04_hgbr


## 10. Analyse des erreurs selon les valeurs manquantes

In [24]:
import numpy as np

# Entraîner HGBR sur tout le train pour analyser les résidus
hgbr_full = HistGradientBoostingRegressor(random_state=0)
hgbr_full.fit(X, y)
preds = hgbr_full.predict(X)
residuals = pd.Series(np.abs(preds - y.values), index=y.index)

for col in ["on", "off"]:
    mask_missing = X[col].isna()
    mae_missing = residuals[mask_missing].mean()
    mae_present = residuals[~mask_missing].mean()
    print(f"MAE quand '{col}' manquant  : {mae_missing:.4f}  ({mask_missing.sum()} lignes)")
    print(f"MAE quand '{col}' présent   : {mae_present:.4f}  ({(~mask_missing).sum()} lignes)")
    print()

MAE quand 'on' manquant  : 4.6214  (13220 lignes)
MAE quand 'on' présent   : 6.1986  (31370 lignes)

MAE quand 'off' manquant  : 6.4256  (18913 lignes)
MAE quand 'off' présent   : 5.2194  (25677 lignes)



## 11. Conclusion

**Points clés :**
- `HistGradientBoostingRegressor` gère nativement les valeurs manquantes sans imputation.
- La comparaison avec Ridge sur les mêmes folds GroupKFold garantit une comparaison équitable.
- Si HGBR bat Ridge de façon stable sur tous les folds, il devient le nouveau modèle de référence.
- L'analyse des erreurs selon la présence de `on`/`off` indique si les valeurs manquantes sont informatives.

**Prochaine étape** : `06_tabular_pipeline.ipynb` — intégration des variables catégorielles avec `skrub.tabular_pipeline`.